# Data Audit

Through this notebook, one is going to inspect the data quality of the dataset.

The dataset comes from NYC Taxi Records: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

The dataset is focused on the yellow taxi record of the database.

# Analysis Steps

1. shape
2. head()
3. info()
4. columns
5. dtypes
6. missing values
7. duplicates / uniqueness
8. numerical summaries
9. categorical distributions
10. date/time ranges
11. business-rule checks
12. foreign-key checks
13. outlier investigation
14. data-quality summary
15. define cleaning rules

## Set Up

In [1]:
# All the imports that will be used for this dataset

import pandas as pd
import pyarrow

## Loading the Data

In [28]:
yellow_trip_data = pd.read_parquet("../data/raw/yellow_tripdata_2026-01.parquet")
zones = pd.read_csv("../data/raw/taxi_zone_lookup.csv")

## Confirm that the dataset is loaded correctly

In [3]:
# Inspecting 10 random rows in the dataset

yellow_trip_data.sample(n=10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2017841,2,2026-01-23 21:34:59,2026-01-23 21:38:58,1.0,1.02,1.0,N,237,236,1,7.20,1.0,0.5,2.44,0.0,1.0,14.64,2.5,0.00,0.00
1184874,2,2026-01-14 21:57:33,2026-01-14 22:25:35,2.0,15.76,1.0,N,132,255,1,59.00,1.0,0.5,12.30,0.0,1.0,75.55,0.0,1.75,0.00
2280848,2,2026-01-28 10:20:17,2026-01-28 10:30:47,1.0,0.58,1.0,N,239,142,1,10.70,0.0,0.5,2.94,0.0,1.0,17.64,2.5,0.00,0.00
2456691,1,2026-01-30 06:22:12,2026-01-30 06:31:46,1.0,2.00,1.0,N,263,74,1,12.10,3.5,0.5,3.40,0.0,1.0,20.50,2.5,0.00,0.00
1151995,2,2026-01-14 16:16:27,2026-01-14 16:37:37,1.0,2.39,1.0,N,161,239,1,19.80,2.5,0.5,2.00,0.0,1.0,29.05,2.5,0.00,0.75
920134,1,2026-01-12 08:22:08,2026-01-12 08:50:32,1.0,6.00,99.0,N,155,55,1,27.50,0.0,0.5,0.00,0.0,0.0,28.00,0.0,0.00,0.00
2114386,2,2026-01-24 21:21:04,2026-01-24 21:47:14,1.0,4.24,1.0,N,45,62,1,25.40,1.0,0.5,6.23,0.0,1.0,37.38,2.5,0.00,0.75
1928319,2,2026-01-22 23:05:46,2026-01-22 23:29:24,1.0,8.02,1.0,N,132,265,1,33.10,1.0,0.5,7.00,0.0,1.0,44.35,0.0,1.75,0.00
3411106,2,2026-01-24 19:20:20,2026-01-24 19:29:16,NaN,1.06,NaN,NaN,107,170,0,10.07,0.0,0.5,0.00,0.0,1.0,19.82,NaN,NaN,0.75
3285210,2,2026-01-21 15:07:11,2026-01-21 15:26:02,NaN,2.98,NaN,NaN,17,80,0,24.25,0.0,0.5,0.00,0.0,1.0,25.75,NaN,NaN,0.00


In [4]:
# Inspecting column names, numbers, and also their data types

yellow_trip_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [5]:
# Inspecting the shape of the dataset
yellow_trip_data.shape

(3724889, 20)

## Initial Observations:

- Number of rows: 3724889
- Number of columns: 20
- Unit of observation: an individual taxi trip.
- Initial data quality concerns: There exist some data type concerns in the information tab. This will be further investigated later.

## Column Inventory

In [6]:
yellow_trip_data.columns.tolist()

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

Data Dictionary

* VendorID: A code indicating the TPEP provider that provided the record.
    1 = Creative Mobile Technologies, LLC
    2 = Curb Mobility, LLC
    6 = Myle Technologies Inc
    7 = Helix

* tpep_pickup_datetime : The date and time when the meter was engaged.
* tpep_dropoff_datetime: The date and time when the meter was disengaged. 
* passenger_count: The number of passengers in the vehicle. 
* trip_distance: The elapsed trip distance in miles reported by the taximeter.
* RatecodeID: The final rate code in effect at the end of the trip.
    1 = Standard rate
    2 = JFK
    3 = Newark
    4 = Nassau or Westchester
    5 = Negotiated fare
    6 = Group ride
    99 = Null/unknown
* store_and_fwd_flag: This flag indicates whether the trip record was held in vehicle memory before
    sending to the vendor, aka “store and forward,” because the vehicle did not
    have a connection to the server.
    Y = store and forward trip
    N = not a store and forward trip
* PULocationID: TLC Taxi Zone in which the taximeter was engaged.
* DOLocationID: TLC Taxi Zone in which the taximeter was disengaged.
* payment_type: A numeric code signifying how the passenger paid for the trip.
    0 = Flex Fare trip
    1 = Credit card
    2 = Cash
    3 = No charge
    4 = Dispute
    5 = Unknown
    6 = Voided trip
* fare_amount: The time-and-distance fare calculated by the meter. For additional
    information on the following columns, see
    https://www.nyc.gov/site/tlc/passengers/taxi-fare.page
* extra: Miscellaneous extras and surcharges.
* mta_tax: Tax that is automatically triggered based on the metered rate in use
* tip_amount: Tip amount – This field is automatically populated for credit card tips. Cash
    tips are not included.
* tolls_amount: Total amount of all tolls paid in trip.
* improvement_surcharge: Improvement surcharge assessed trips at the flag drop. The improvement
    surcharge began being levied in 2015.
* total_amount: The total amount charged to passengers. Does not include cash tips.
* congestion_surcharge: Total amount collected in trip for NYS congestion surcharge.
* airport_fee: For pick up only at LaGuardia and John F. Kennedy Airports.
* cbd_congestion_fee: Per-trip charge for MTA's Congestion Relief Zone starting Jan. 5, 2025

## Data Types

In [7]:
yellow_trip_data.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

### Data Types Report

There is one column might not be in the correct data types

* Passenger_count is displayed in float64 (This should be in int32 since passenger is a discrete variable)

## Missing Values

In [8]:
yellow_trip_data.isna().sum()

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [11]:
missing_columns = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

yellow_trip_data[missing_columns].isna().all(axis=1).sum()

np.int64(1088058)

### Missing Values Report:

There are five columns that have missing values:
* passenger_count
* RatecodeID 
* store_and_fwd_flag
* congestion_surcharge
* Airport_fee

They all miss same amount of values which is 1088058 rows. This means that there is a missing pattern in the data.

# Check for duplicates & Uniqueness

In [18]:
# Check for duplicates in the data

duplicated_count = yellow_trip_data.duplicated().sum()
duplicated_count

np.int64(0)

Duplicate check has been passed since there are no exact same observations across different rows.

In [ ]:
# Check for uniqueness in the data

yellow_trip_data.nunique()

VendorID                       4
tpep_pickup_datetime     1757191
tpep_dropoff_datetime    1755995
passenger_count               10
trip_distance               4777
RatecodeID                     7
store_and_fwd_flag             2
PULocationID                 262
DOLocationID                 260
payment_type                   5
fare_amount                12867
extra                         45
mta_tax                        6
tip_amount                  4326
tolls_amount                1495
improvement_surcharge          4
total_amount               21228
congestion_surcharge           3
Airport_fee                    8
cbd_congestion_fee             3
dtype: int64

In [ ]:
# Check for passenger Count in the data

yellow_trip_data["passenger_count"].value_counts()

passenger_count
1.0    2150994
2.0     334370
3.0      72864
4.0      49738
0.0      14787
5.0       9184
6.0       4887
8.0          4
7.0          2
9.0          1
Name: count, dtype: int64

In [17]:
yellow_trip_data[yellow_trip_data['passenger_count'] > 4]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
312,2,2026-01-01 00:09:20,2026-01-01 00:13:14,5.0,0.59,1.0,N,238,239,1,5.8,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,0.00
334,2,2026-01-01 00:17:54,2026-01-01 00:29:58,5.0,1.17,1.0,N,107,170,1,12.1,1.0,0.5,3.00,0.0,1.0,20.85,2.5,0.0,0.75
335,2,2026-01-01 00:48:18,2026-01-01 00:54:29,5.0,1.01,1.0,N,114,107,1,7.9,1.0,0.5,4.10,0.0,1.0,17.75,2.5,0.0,0.75
684,2,2026-01-01 00:28:24,2026-01-01 00:41:05,5.0,1.86,1.0,N,142,236,1,13.5,1.0,0.5,3.70,0.0,1.0,22.20,2.5,0.0,0.00
685,2,2026-01-01 00:42:26,2026-01-01 01:12:47,5.0,5.21,1.0,N,43,249,1,30.3,1.0,0.5,5.00,0.0,1.0,41.05,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2633979,2,2026-01-31 23:06:05,2026-01-31 23:24:37,6.0,2.90,1.0,N,90,237,2,19.1,1.0,0.5,0.00,0.0,1.0,24.85,2.5,0.0,0.75
2635206,2,2026-01-31 23:09:36,2026-01-31 23:26:23,6.0,7.20,1.0,N,142,244,1,30.3,1.0,0.5,7.21,0.0,1.0,43.26,2.5,0.0,0.75
2635306,2,2026-01-31 23:19:32,2026-01-31 23:26:51,5.0,0.66,1.0,N,79,4,1,7.9,1.0,0.5,3.00,0.0,1.0,16.65,2.5,0.0,0.75
2635307,2,2026-01-31 23:28:16,2026-01-31 23:34:30,5.0,0.70,1.0,N,4,79,1,7.2,1.0,0.5,2.59,0.0,1.0,15.54,2.5,0.0,0.75


There are interesting things with the passenger count happening in the data. Legally, with the driver counted there can be only 4 passenger max in a yellow cab and there are 14078 observations that have more than 4 passengers in the cab which means that there are some outliers in the data.

## Numerical Summaries

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.2f}") # Change the display format to two decimals

yellow_trip_data.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
count,"3,724,889.00",3724889,3724889,"2,636,831.00","3,724,889.00","2,636,831.00","3,724,889.00","3,724,889.00","3,724,889.00","3,724,889.00","3,724,889.00","3,724,889.00","3,724,889.00","3,724,889.00","3,724,889.00","3,724,889.00","2,636,831.00","2,636,831.00","3,724,889.00"
mean,1.87,2026-01-17 01:44:52.106518,2026-01-17 02:02:03.696138,1.26,6.46,5.22,161.44,160.99,0.85,20.80,1.02,0.48,2.61,0.50,0.95,29.18,2.15,0.15,0.52
min,1.00,2025-12-31 23:57:29,2025-12-31 23:57:32,0.00,0.00,1.00,1.00,1.00,0.00,"-2,555.20",-7.50,-0.50,-88.88,-94.50,-1.00,"-2,560.20",-2.50,-1.75,-0.75
25%,2.00,2026-01-09 17:51:59,2026-01-09 18:08:47,1.00,1.00,1.00,114.00,107.00,0.00,10.00,0.00,0.50,0.00,0.00,1.00,17.00,2.50,0.00,0.00
50%,2.00,2026-01-16 21:20:56,2026-01-16 21:36:14,1.00,1.81,1.00,161.00,162.00,1.00,15.60,0.00,0.50,2.00,0.00,1.00,23.05,2.50,0.00,0.75
75%,2.00,2026-01-24 07:25:11,2026-01-24 07:40:22,1.00,3.73,1.00,233.00,234.00,1.00,26.10,2.50,0.50,3.71,0.00,1.00,33.83,2.50,0.00,0.75
max,7.00,2026-02-01 00:45:01,2026-02-01 23:35:31,9.00,"269,097.48",99.00,265.00,265.00,4.00,"2,555.20",17.46,4.75,766.00,122.22,1.00,"2,560.20",2.50,26.75,0.75
std,0.70,NaN,NaN,0.67,648.89,19.65,67.07,71.04,0.71,18.93,1.71,0.11,3.92,2.13,0.27,22.59,0.94,0.53,0.36


Interesting time is that the pick up time includes 2025-12-31 and also 2026-02-01. The dataset description should only include rides that are within Janurary of 2026 but here we are finding data that are out of bounds.

There are also trip distance that are zero miles which requires further investigation. And also the largest trip distance is very large (269,097.48) which requires further investigation.

The minimum fare_amount if negative -2555.20 which requires further investigation.

Same thing applies to extra, mta_tax, tip_amount, improvement_surcharge, total_amount, congestion_surcharge, Airport_free, cbd_congestion_fee which are negative numbers which logically does not make sense.

There needs to be more investigation into the minimum observations of these data.

## Business Rule Check

In [31]:
zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [32]:
invalid_pickup = (
    set(yellow_trip_data["PULocationID"].dropna()) - set(zones["LocationID"].dropna())
)

invalid_dropoff = (
    set(yellow_trip_data["DOLocationID"].dropna()) - set(zones["LocationID"].dropna())
)

invalid_pickup, invalid_dropoff

(set(), set())

Foreign key integrity check has been passed. There are no invalid pick up zones nor invalid drop off zones.